# 2D Landmark SLAM — minimal GTSAM example

One robot (`o1`, `x1..x3`) drives along +x and observes two landmarks (`l1`, `l2`)
with bearing-range measurements. Jointly optimises poses + landmarks.

This deliberately tiny graph is exactly solvable; real landmark data contains redundant, noisy observations.

Tested with `gtsam==4.3.0`.


In [ ]:
import gtsam
import numpy as np
import matplotlib.pyplot as plt
from gtsam.utils.plot import plot_pose2_on_axes


In [ ]:
graph = gtsam.NonlinearFactorGraph()
initial = gtsam.Values()

# Prior on the first pose. Use a tiny sigma (not exact zero) for numerical stability.
prior_mean = gtsam.Pose2(0, 0, 0)
prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-6, 1e-6, 1e-8]))
o1 = gtsam.symbol("o", 1)
graph.add(gtsam.PriorFactorPose2(o1, prior_mean, prior_noise))
initial.insert(o1, prior_mean)

# Odometry: 2 m steps along +x.
odom_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.2, 0.2, 0.1]))
x1, x2, x3 = (gtsam.symbol("x", i) for i in (1, 2, 3))
graph.add(gtsam.BetweenFactorPose2(o1, x1, gtsam.Pose2(2, 0, 0), odom_noise))
initial.insert(x1, gtsam.Pose2(2, 0, 0))
graph.add(gtsam.BetweenFactorPose2(x1, x2, gtsam.Pose2(2, 0, 0), odom_noise))
initial.insert(x2, gtsam.Pose2(4, 0, 0))
graph.add(gtsam.BetweenFactorPose2(x2, x3, gtsam.Pose2(2, 0, 0), odom_noise))
initial.insert(x3, gtsam.Pose2(6, 0, 0))

# Bearing-range observations: bearing Rot2(0) (straight ahead in robot frame),
# range 2 m. True landmarks are at (4, 0) and (6, 0); initials are offset on purpose.
lm_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.3, 0.3]))
l1, l2 = gtsam.symbol("l", 1), gtsam.symbol("l", 2)
graph.add(gtsam.BearingRangeFactor2D(x1, l1, gtsam.Rot2(0), 2.0, lm_noise))
initial.insert(l1, gtsam.Point2(2, 1))
graph.add(gtsam.BearingRangeFactor2D(x2, l2, gtsam.Rot2(0), 2.0, lm_noise))
initial.insert(l2, gtsam.Point2(4, 1))

print(f"Graph with {graph.size()} factors, {initial.size()} initial values")

In [ ]:
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial)
result = optimizer.optimize()
print(f"Initial error: {graph.error(initial):.3f} -> final: {graph.error(result):.3f}")
result.print("Optimised values:\n")

In [ ]:
# Plot poses (red/green axes) and landmarks (blue *) on shared axes.
# GTSAM 4.x: use `plot_pose2_on_axes(ax, ...)` — the old `axis=` kwarg was removed.
fig, ax = plt.subplots()
for sym in (o1, x1, x2, x3):
    plot_pose2_on_axes(ax, result.atPose2(sym), 0.5)
for sym in (l1, l2):
    pt = result.atPoint2(sym)
    ax.plot(pt[0], pt[1], "b*", markersize=12)
ax.set_aspect("equal")
ax.set_title("Poses + landmarks after optimisation")
plt.show()
